# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get metadata as dict for easy printing (not for attribute access)
metadata_json = dataset.metadata.to_json()
print(f"Dataset Title: {metadata_json.get('name')}")
print(f"Description: {metadata_json.get('description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all RecordSets, their @id, and associated Field @ids
print("Available record sets and fields:")
record_set_ids = []
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet: {rs.name} (ID: {rs.id})")
    record_set_ids.append(rs.id)
    print("    Fields:")
    for field in rs.fields:
        print(f"        - {field.name} (ID: {field.id}, dataType: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# List all records sets' @id
print("RecordSet @ids:")
for i, rs_id in enumerate(record_set_ids):
    print(f"  {i}: {rs_id}")

# Choose the primary record set for protein data: The main @id will usually include "protein", "main", or similar, but let's just pick the first.
record_set_id = record_set_ids[0]

# Load all records for each record set into a dict of DataFrames
dataframes = {}
for rs_id in record_set_ids:
    # Use '@id' for the record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns in the main DataFrame
print(f"Columns in record set {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field by inspecting columns
df = dataframes[record_set_id]
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]

if not numeric_candidates:
    # Try to guess numeric columns even if loaded as object
    possible_numeric_fields = ['coverage', 'peptide_count', 'MW', 'pI', 'abundance', 'score']
    for col in df.columns:
        if any(x.lower() in col.lower() for x in possible_numeric_fields):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except Exception:
                pass
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]

if numeric_candidates:
    numeric_field = numeric_candidates[0]  # choose the first candidate
else:
    # fallback: dataset may contain numeric values under string columns. Try conversion manually.
    print("No obvious numeric field found for EDA.")
    numeric_field = df.columns[0]

print(f"Using numeric field for analysis: {numeric_field}")

threshold = df[numeric_field].mean() if df[numeric_field].mean() is not None else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize this numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g. 'description' or 'sample_group' if present
possible_group_fields = ['description', 'sample_group', 'accession', 'name']
group_field = None
for field in possible_group_fields:
    if field in filtered_df.columns:
        group_field = field
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())
else:
    print("No group field found for grouping analysis.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If a grouping field exists, plot mean values
if group_field:
    grouped_df_sorted = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.figure(figsize=(10,4))
    sns.barplot(x=grouped_df_sorted.index, y=numeric_field, data=grouped_df_sorted)
    plt.title(f"Top 10 {group_field}s by mean {numeric_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key findings:**
- The dataset provides mass spectrometry-derived information about extracellular vesicle proteins from stimulated human mast cells.
- Main fields include accession, description, coverage, counts, molecular weight, and other protein properties, accessible via their respective `@id`.
- The data contains numeric variability, and top proteins/groups can be identified by abundance or other chosen attributes.
- The notebook provides an extensible pattern for further analysis and visualization using mlcroissant-compliant Croissant schema datasets.